In [0]:
# Databricks notebook source

# UC3 - RAG Evaluation: LLM as a Judge
# MediLife Insurance - primeinsurance_genai.gold.rag_query_history
#
# This notebook:
#   1. Reads the latest 5 queries from rag_query_history
#   2. Reconstructs ground truth from dim_policy for each query
#   3. Runs three LLM-as-a-judge evaluations:
#        - Faithfulness  : did the LLM only use what was in the retrieved context?
#        - Correctness   : is the answer actually right against the source table?
#        - Completeness  : did the LLM answer the full question or dodge parts of it?
#   4. Prints a per-query scorecard and overall averages
#   5. Writes results to primeinsurance_genai.gold.rag_eval_results



In [0]:
from openai import OpenAI
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc
from pyspark.sql.types import (
    StructType, StructField,
    StringType, FloatType, TimestampType
)

import pandas as pd
import json
import re
from datetime import datetime, timezone

spark = SparkSession.builder.getOrCreate()

In [0]:
# COMMAND ----------

# Config

CATALOG        = "primeinsurance_genai"
SCHEMA         = "gold"
HISTORY_TABLE  = f"{CATALOG}.{SCHEMA}.rag_query_history_v2"
SOURCE_TABLE   = f"{CATALOG}.{SCHEMA}.dim_policy"
EVAL_TABLE     = f"{CATALOG}.{SCHEMA}.rag_eval_results"

DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
WORKSPACE_URL    = spark.conf.get("spark.databricks.workspaceUrl")

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"https://{WORKSPACE_URL}/serving-endpoints"
)

LLM_MODEL = "databricks-meta-llama-3-3-70b-instruct"

print("Config loaded.")

In [0]:
# COMMAND ----------

# ── SECTION 1: Load history table - latest run per question only ──────────────
#
# Use a window function to deduplicate - keep only the most recent row
# per question so repeated notebook runs do not skew the evaluation.

w = Window.partitionBy("question").orderBy(desc("retrieved_at"))

history_pd = (
    spark.table(HISTORY_TABLE)
    .withColumn("rn", row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .toPandas()
)

print(f"Queries loaded for evaluation: {len(history_pd)}")
print()
for _, row in history_pd.iterrows():
    print(f"  Q: {row['question'][:80]}")
    print(f"     Sources: {row['source_policies']}")
    print(f"     Confidence: {row['confidence']}")
    print()

In [0]:
# COMMAND ----------

# ── SECTION 2: Load source table for ground truth ─────────────────────────────
#
# The source table is the ground truth for this evaluation.
# Because dim_policy is structured, we can verify answers exactly
# rather than relying entirely on the judge's opinion.
# This is a significant advantage over typical RAG evaluation on unstructured data.

policy_pd = (
    spark.table(SOURCE_TABLE)
    .filter(F.col("is_current") == True)
    .toPandas()
)

# Convert policy_number to string for consistent matching
policy_pd["policy_number"] = policy_pd["policy_number"].astype(str)

print(f"Source policies loaded: {len(policy_pd):,}")

In [0]:
# COMMAND ----------

# ── SECTION 3: Ground truth builder ──────────────────────────────────────────
#
# For each question type we extract the actual values from dim_policy.
# This ground truth is passed to the judge so it can compare the LLM answer
# against what the database actually says, not just against the retrieved chunks.

def build_ground_truth(question: str, policy_df: pd.DataFrame) -> str:
    """
    Build a ground truth string by querying the source table directly.

    Supports four query patterns:
      - Exact policy number lookup
      - Umbrella coverage filter
      - State-based filter
      - Deductible comparison
    Falls back to "not verifiable" for open-ended semantic queries.
    """
    q_lower = question.lower()

    # Exact policy number lookup
    named = re.findall(r'\b\d{5,7}\b', question)
    if named:
        rows = policy_df[policy_df["policy_number"].isin(named)]
        if len(rows) > 0:
            r = rows.iloc[0]
            umbrella = (
                f"YES - ${int(r['umbrella_limit']):,}"
                if not r["_umbrella_limit_zero_flag"]
                else "NO"
            )
            return (
                f"Policy {r['policy_number']}: "
                f"State={r['policy_state_full']}, "
                f"CSL={r['policy_csl']}, "
                f"Deductible=${int(r['policy_deductable']):,}, "
                f"Premium=${float(r['policy_annual_premium']):,.2f}, "
                f"Umbrella={umbrella}"
            )
        return f"Policy {named[0]} does not exist in the table."

    # Umbrella filter
    if "umbrella" in q_lower:
        umbrella_yes = policy_df[policy_df["_umbrella_limit_zero_flag"] == False]
        umbrella_no  = policy_df[policy_df["_umbrella_limit_zero_flag"] == True]
        sample       = umbrella_yes.head(5)
        lines = [
            f"Policy {r['policy_number']}: umbrella limit ${int(r['umbrella_limit']):,}"
            for _, r in sample.iterrows()
        ]
        return (
            f"Total policies WITH umbrella coverage: {len(umbrella_yes)}\n"
            f"Total policies WITHOUT umbrella coverage: {len(umbrella_no)}\n"
            f"Sample umbrella policies:\n" + "\n".join(lines)
        )

    # State filter
    state_map = {
        "illinois": "IL", "ohio": "OH", "indiana": "IN",
        "new york": "NY", "west virginia": "WV", "virginia": "VA",
        "north carolina": "NC", "south carolina": "SC"
    }
    for state_name, state_code in state_map.items():
        if state_name in q_lower or f" {state_code.lower()} " in q_lower:
            state_df   = policy_df[policy_df["policy_state_code"] == state_code]
            csl_counts = state_df["policy_csl"].value_counts().to_dict()
            umbrella   = state_df[state_df["_umbrella_limit_zero_flag"] == False]
            return (
                f"{state_name.title()} policies: {len(state_df)} total\n"
                f"CSL distribution: {csl_counts}\n"
                f"Policies with umbrella: {len(umbrella)}"
            )

    # Deductible comparison
    if "500" in question and "2000" in question:
        d500  = policy_df[policy_df["policy_deductable"] == 500]
        d2000 = policy_df[policy_df["policy_deductable"] == 2000]
        return (
            f"$500 deductible: {len(d500)} policies, "
            f"avg premium ${d500['policy_annual_premium'].mean():.2f}, "
            f"range ${d500['policy_annual_premium'].min():.2f} - ${d500['policy_annual_premium'].max():.2f}\n"
            f"$2000 deductible: {len(d2000)} policies, "
            f"avg premium ${d2000['policy_annual_premium'].mean():.2f}, "
            f"range ${d2000['policy_annual_premium'].min():.2f} - ${d2000['policy_annual_premium'].max():.2f}"
        )

    return "No specific ground truth available for this query type."


def build_context_from_sources(source_policies: list, policy_df: pd.DataFrame) -> str:
    """
    Reconstruct the approximate context that was passed to the LLM
    by looking up the source policy numbers in the current table.
    Used for faithfulness evaluation.
    """
    rows = policy_df[policy_df["policy_number"].isin(source_policies)]
    parts = []
    for _, r in rows.iterrows():
        umbrella = (
            f"HAS umbrella coverage - limit ${int(r['umbrella_limit']):,}"
            if not r["_umbrella_limit_zero_flag"]
            else "does NOT have umbrella coverage"
        )
        parts.append(
            f"Policy {r['policy_number']}: "
            f"State={r['policy_state_full']} ({r['policy_state_code']}), "
            f"CSL={r['policy_csl']}, "
            f"Deductible=${int(r['policy_deductable']):,}, "
            f"Premium=${float(r['policy_annual_premium']):,.2f}, "
            f"Umbrella: {umbrella}"
        )
    return "\n".join(parts)

In [0]:

# COMMAND ----------

# ── SECTION 4: Judge prompts ───────────────────────────────────────────────────

FAITHFULNESS_PROMPT = """You are evaluating an AI answer for faithfulness to its source context.

Faithfulness means: every factual claim in the answer must be directly supported by the context.
The answer should NOT include facts that are absent from or contradict the context.

SCORING:
1.0 = every factual claim is supported by the context
0.5 = some claims supported, some unsupported or contradicted
0.0 = answer contains claims not supported by the context, or contradicts it

Respond ONLY with valid JSON, no markdown, no explanation outside the JSON:
{{
  "faithfulness_score": <float 0.0 to 1.0>,
  "supported_claims": ["claim 1", "claim 2"],
  "unsupported_claims": ["claim 1"],
  "reasoning": "one sentence"
}}

--- CONTEXT (what was retrieved and given to the AI) ---
{context}

--- ANSWER (what the AI said) ---
{answer}
"""

CORRECTNESS_PROMPT = """You are evaluating an AI answer for factual correctness against database ground truth.

Correctness means: the facts stated in the answer match the actual values in the database.

SCORING:
1.0 = all stated facts are correct and match the ground truth
0.5 = partially correct, some facts right some wrong or missing
0.0 = answer states incorrect facts, or refuses to answer when ground truth is available

Respond ONLY with valid JSON, no markdown, no explanation outside the JSON:
{{
  "correctness_score": <float 0.0 to 1.0>,
  "correct_facts": ["fact 1"],
  "incorrect_facts": ["fact 1"],
  "reasoning": "one sentence"
}}

--- QUESTION ---
{question}

--- ANSWER ---
{answer}

--- GROUND TRUTH (actual database values) ---
{ground_truth}
"""

COMPLETENESS_PROMPT = """You are evaluating whether an AI answer fully addresses the question asked.

Completeness means: the answer addresses every part of the question without dodging or leaving gaps.

SCORING:
1.0 = question is fully answered, nothing missing
0.5 = question is partially answered, some parts addressed some ignored
0.0 = answer refuses to answer, says only "I don't know", or addresses a completely different question

Respond ONLY with valid JSON, no markdown, no explanation outside the JSON:
{{
  "completeness_score": <float 0.0 to 1.0>,
  "answered_parts": ["part 1"],
  "unanswered_parts": ["part 1"],
  "reasoning": "one sentence"
}}

--- QUESTION ---
{question}

--- ANSWER ---
{answer}
"""

In [0]:

# COMMAND ----------

# ── SECTION 5: Judge caller ────────────────────────────────────────────────────

def call_judge(prompt: str) -> dict:
    """
    Call the LLM judge with a given prompt and parse the JSON response.
    Strips markdown code fences if the model wraps the JSON in them.
    Returns the parsed dict or an error dict if parsing fails.
    """
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=600,
        )
        raw = response.choices[0].message.content.strip()
        # Strip markdown fences if present
        raw = re.sub(r"```json|```", "", raw).strip()
        return json.loads(raw)
    except json.JSONDecodeError as e:
        return {"error": f"JSON parse failed: {e}", "raw": raw}
    except Exception as e:
        return {"error": str(e)}

# COMMAND ----------

# ── SECTION 6: Run evaluation for all queries ──────────────────────────────────

eval_results = []

for _, row in history_pd.iterrows():
    question        = row["question"]
    answer          = row["answer"]
    source_policies = row["source_policies"]
    query_id        = row["query_id"]
    confidence      = float(row["confidence"])

    print(f"Evaluating: {question[:70]}...")

    # Build context and ground truth
    context      = build_context_from_sources(source_policies, policy_pd)
    ground_truth = build_ground_truth(question, policy_pd)

    # Run all three judges
    faith_result = call_judge(
        FAITHFULNESS_PROMPT.format(context=context, answer=answer)
    )
    corr_result = call_judge(
        CORRECTNESS_PROMPT.format(question=question, answer=answer, ground_truth=ground_truth)
    )
    comp_result = call_judge(
        COMPLETENESS_PROMPT.format(question=question, answer=answer)
    )

    # Extract scores safely - default to None if judge call failed
    faithfulness  = faith_result.get("faithfulness_score")
    correctness   = corr_result.get("correctness_score")
    completeness  = comp_result.get("completeness_score")

    # Compute overall score as average of the three dimensions
    scores = [s for s in [faithfulness, correctness, completeness] if s is not None]
    overall = round(sum(scores) / len(scores), 4) if scores else None

    eval_results.append({
        "query_id":              query_id,
        "question":              question,
        "answer":                answer,
        "confidence":            confidence,
        # scores
        "faithfulness_score":    faithfulness,
        "correctness_score":     correctness,
        "completeness_score":    completeness,
        "overall_score":         overall,
        # reasoning from each judge
        "faithfulness_reasoning": faith_result.get("reasoning"),
        "correctness_reasoning":  corr_result.get("reasoning"),
        "completeness_reasoning": comp_result.get("completeness_reasoning",
                                                   comp_result.get("reasoning")),
        # supporting detail
        "unsupported_claims":    str(faith_result.get("unsupported_claims", [])),
        "incorrect_facts":       str(corr_result.get("incorrect_facts", [])),
        "unanswered_parts":      str(comp_result.get("unanswered_parts", [])),
        "ground_truth_used":     ground_truth,
        "evaluated_at":          datetime.now(timezone.utc),
    })

    print(f"  Faithfulness : {faithfulness}  - {faith_result.get('reasoning', 'N/A')}")
    print(f"  Correctness  : {correctness}  - {corr_result.get('reasoning', 'N/A')}")
    print(f"  Completeness : {completeness}  - {comp_result.get('reasoning', 'N/A')}")
    print(f"  Overall      : {overall}")
    print()

eval_df = pd.DataFrame(eval_results)

In [0]:
# COMMAND ----------

# ── SECTION 7: Print scorecard ─────────────────────────────────────────────────

print("=" * 80)
print("RAG EVALUATION SCORECARD")
print("=" * 80)

scorecard_cols = [
    "question", "confidence",
    "faithfulness_score", "correctness_score",
    "completeness_score", "overall_score"
]

display_df = eval_df[scorecard_cols].copy()
display_df["question"] = display_df["question"].str[:55]

print(display_df.to_string(index=False))

print()
print("─" * 80)
print("AVERAGES")
print("─" * 80)

for col in ["faithfulness_score", "correctness_score", "completeness_score", "overall_score"]:
    vals = eval_df[col].dropna()
    print(f"  {col:30s}: {vals.mean():.3f}  (n={len(vals)})")

print()
print("─" * 80)
print("NOTES")
print("─" * 80)
print("  Faithfulness : did the LLM stay within the retrieved context?")
print("  Correctness  : does the answer match actual values in dim_policy?")
print("  Completeness : did the LLM answer the full question?")
print("  Overall      : simple average of the three dimensions")
print()
print("  Confidence is the RAG pipeline's own heuristic (cosine similarity based).")
print("  Compare it against the judge scores to see if it is well-calibrated.")

In [0]:
# COMMAND ----------

# ── SECTION 8: Write eval results to Delta ─────────────────────────────────────

eval_schema = StructType([
    StructField("query_id",                StringType(),  False),
    StructField("question",                StringType(),  True),
    StructField("answer",                  StringType(),  True),
    StructField("confidence",              FloatType(),   True),
    StructField("faithfulness_score",      FloatType(),   True),
    StructField("correctness_score",       FloatType(),   True),
    StructField("completeness_score",      FloatType(),   True),
    StructField("overall_score",           FloatType(),   True),
    StructField("faithfulness_reasoning",  StringType(),  True),
    StructField("correctness_reasoning",   StringType(),  True),
    StructField("completeness_reasoning",  StringType(),  True),
    StructField("unsupported_claims",      StringType(),  True),
    StructField("incorrect_facts",         StringType(),  True),
    StructField("unanswered_parts",        StringType(),  True),
    StructField("ground_truth_used",       StringType(),  True),
    StructField("evaluated_at",            TimestampType(), True),
])

eval_sdf = spark.createDataFrame(
    [
        (
            r["query_id"],
            r["question"],
            r["answer"],
            float(r["confidence"]) if r["confidence"] is not None else None,
            float(r["faithfulness_score"])  if r["faithfulness_score"]  is not None else None,
            float(r["correctness_score"])   if r["correctness_score"]   is not None else None,
            float(r["completeness_score"])  if r["completeness_score"]  is not None else None,
            float(r["overall_score"])       if r["overall_score"]       is not None else None,
            r["faithfulness_reasoning"],
            r["correctness_reasoning"],
            r["completeness_reasoning"],
            r["unsupported_claims"],
            r["incorrect_facts"],
            r["unanswered_parts"],
            r["ground_truth_used"],
            r["evaluated_at"],
        )
        for r in eval_results
    ],
    schema=eval_schema
)

eval_sdf.write.format("delta").mode("overwrite").saveAsTable(EVAL_TABLE)
print(f"Eval results written to {EVAL_TABLE}")

# COMMAND ----------

# ── SECTION 9: Verify eval table ──────────────────────────────────────────────

spark.table(EVAL_TABLE).select(
    "query_id",
    "question",
    "faithfulness_score",
    "correctness_score",
    "completeness_score",
    "overall_score",
    "evaluated_at"
).show(truncate=60)